In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EcommerceAnalytics")
    .master("local[*]")
    .getOrCreate()
)

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 11:20:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.0.5


In [3]:
data = [
    (101, "Laptop", 50000),
    (102, "Mobile", 20000),
    (103, "Laptop", 60000),
    (104, "Tablet", 15000)
]

columns = ["order_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     101| Laptop| 50000|
|     102| Mobile| 20000|
|     103| Laptop| 60000|
|     104| Tablet| 15000|
+--------+-------+------+



In [4]:
#filter
df.filter(df.amount>2000).show()

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     101| Laptop| 50000|
|     102| Mobile| 20000|
|     103| Laptop| 60000|
|     104| Tablet| 15000|
+--------+-------+------+



In [5]:
#Group by
df.groupBy("product").sum("amount").show()

+-------+-----------+
|product|sum(amount)|
+-------+-----------+
| Laptop|     110000|
| Mobile|      20000|
| Tablet|      15000|
+-------+-----------+



In [6]:
#Sorting
df.orderBy(df.amount.desc()).show()

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     103| Laptop| 60000|
|     101| Laptop| 50000|
|     102| Mobile| 20000|
|     104| Tablet| 15000|
+--------+-------+------+



In [7]:
#count
print(df.count())

4


In [8]:
df.show(3)

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     101| Laptop| 50000|
|     102| Mobile| 20000|
|     103| Laptop| 60000|
+--------+-------+------+
only showing top 3 rows


In [9]:
df.select('product','amount').show()

+-------+------+
|product|amount|
+-------+------+
| Laptop| 50000|
| Mobile| 20000|
| Laptop| 60000|
| Tablet| 15000|
+-------+------+



In [10]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)



In [11]:
df.filter(df.amount>10000).show()

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     101| Laptop| 50000|
|     102| Mobile| 20000|
|     103| Laptop| 60000|
|     104| Tablet| 15000|
+--------+-------+------+



In [12]:
df.where(df.amount>10000).show()

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     101| Laptop| 50000|
|     102| Mobile| 20000|
|     103| Laptop| 60000|
|     104| Tablet| 15000|
+--------+-------+------+



In [13]:
#products costing more than 10000 AND are Laptop

df.filter((df.product=='Laptop') & (df.amount>10000)).show()

+--------+-------+------+
|order_id|product|amount|
+--------+-------+------+
|     101| Laptop| 50000|
|     103| Laptop| 60000|
+--------+-------+------+



In [14]:
#adding column gst_amount
from pyspark.sql.functions import col
df = df.withColumn('gst_amount', col("amount") * 0.25)
df.show()

+--------+-------+------+----------+
|order_id|product|amount|gst_amount|
+--------+-------+------+----------+
|     101| Laptop| 50000|   12500.0|
|     102| Mobile| 20000|    5000.0|
|     103| Laptop| 60000|   15000.0|
|     104| Tablet| 15000|    3750.0|
+--------+-------+------+----------+



In [15]:
# Create another column 'total_amount'
df = df.withColumn('total_amount', col('amount')+col('gst_amount'))
df.show()

+--------+-------+------+----------+------------+
|order_id|product|amount|gst_amount|total_amount|
+--------+-------+------+----------+------------+
|     101| Laptop| 50000|   12500.0|     62500.0|
|     102| Mobile| 20000|    5000.0|     25000.0|
|     103| Laptop| 60000|   15000.0|     75000.0|
|     104| Tablet| 15000|    3750.0|     18750.0|
+--------+-------+------+----------+------------+



In [16]:
#Perform grouping and aggregation
from pyspark.sql.functions import (sum, avg, count, min, max)




In [17]:
df.groupBy("product").agg(
    sum("amount").alias("total_sales")
).show()

+-------+-----------+
|product|total_sales|
+-------+-----------+
| Laptop|     110000|
| Mobile|      20000|
| Tablet|      15000|
+-------+-----------+



In [18]:
df.groupBy("product").agg(
    sum("amount").alias("total_sales"),
    avg("amount").alias("average_sales"),
    count("order_id").alias("total_orders"),
    max("amount").alias("highest_order"),
    min("amount").alias("lowest_order")).show()

+-------+-----------+-------------+------------+-------------+------------+
|product|total_sales|average_sales|total_orders|highest_order|lowest_order|
+-------+-----------+-------------+------------+-------------+------------+
| Laptop|     110000|      55000.0|           2|        60000|       50000|
| Mobile|      20000|      20000.0|           1|        20000|       20000|
| Tablet|      15000|      15000.0|           1|        15000|       15000|
+-------+-----------+-------------+------------+-------------+------------+



In [19]:
#Sort data
df.orderBy(col("amount").desc()).show()

+--------+-------+------+----------+------------+
|order_id|product|amount|gst_amount|total_amount|
+--------+-------+------+----------+------------+
|     103| Laptop| 60000|   15000.0|     75000.0|
|     101| Laptop| 50000|   12500.0|     62500.0|
|     102| Mobile| 20000|    5000.0|     25000.0|
|     104| Tablet| 15000|    3750.0|     18750.0|
+--------+-------+------+----------+------------+



## Day 2 - Practice Exercises


In [20]:
# Display only order_id and product columns

df.select("order_id","product").show()

+--------+-------+
|order_id|product|
+--------+-------+
|     101| Laptop|
|     102| Mobile|
|     103| Laptop|
|     104| Tablet|
+--------+-------+



In [21]:
#find all the products with an amount greater than 5000

df.filter(df.amount > 5000).show()

+--------+-------+------+----------+------------+
|order_id|product|amount|gst_amount|total_amount|
+--------+-------+------+----------+------------+
|     101| Laptop| 50000|   12500.0|     62500.0|
|     102| Mobile| 20000|    5000.0|     25000.0|
|     103| Laptop| 60000|   15000.0|     75000.0|
|     104| Tablet| 15000|    3750.0|     18750.0|
+--------+-------+------+----------+------------+



In [22]:
# Calculate the total number of orders in each category
from pyspark.sql.functions import count,avg
df.groupBy("product").agg(count("order_id").alias("total_orders")).show()

+-------+------------+
|product|total_orders|
+-------+------------+
| Laptop|           2|
| Mobile|           1|
| Tablet|           1|
+-------+------------+



In [23]:
# Calculate the average order amount for each category

df.groupBy("product").agg(avg("amount").alias("avg_order_amt")).show()

+-------+-------------+
|product|avg_order_amt|
+-------+-------------+
| Laptop|      55000.0|
| Mobile|      20000.0|
| Tablet|      15000.0|
+-------+-------------+



In [24]:
# Display top 3 products based on amount, in descending order

df.orderBy(df.amount.desc()).show(3)

+--------+-------+------+----------+------------+
|order_id|product|amount|gst_amount|total_amount|
+--------+-------+------+----------+------------+
|     103| Laptop| 60000|   15000.0|     75000.0|
|     101| Laptop| 50000|   12500.0|     62500.0|
|     102| Mobile| 20000|    5000.0|     25000.0|
+--------+-------+------+----------+------------+
only showing top 3 rows


## Day 3 - Reading CSV Files

In [25]:
df = spark.read.csv("../data/sample_orders.csv",
                    header = True,
                    inferSchema = True)

df.show()

+--------+-----------+----------+-----------+------+------------+
|order_id|customer_id|   product|   category|amount|order_status|
+--------+-----------+----------+-----------+------+------------+
|     101|       C001|    Laptop|Electronics| 50000|   Delivered|
|     102|       C002|    Mobile|Electronics| 20000|   Delivered|
|     103|       C003|     Shoes|    Fashion|  3000|     Pending|
|     104|       C004|    Tablet|Electronics| 15000|   Delivered|
|     105|       C005|     Shirt|    Fashion|  2000|   Cancelled|
|     106|       C006|Headphones|Electronics|  NULL|   Delivered|
|     107|       C007|     Watch|    Fashion|  5000|     Pending|
|     108|       C008|    Mobile|Electronics| 20000|   Delivered|
|     108|       C008|    Mobile|Electronics| 20000|   Delivered|
|     109|       C009|  Keyboard|Electronics|  2500|        NULL|
+--------+-----------+----------+-----------+------+------------+



In [26]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_status: string (nullable = true)



In [27]:
df.dtypes

[('order_id', 'int'),
 ('customer_id', 'string'),
 ('product', 'string'),
 ('category', 'string'),
 ('amount', 'int'),
 ('order_status', 'string')]

In [28]:
df.columns

['order_id', 'customer_id', 'product', 'category', 'amount', 'order_status']

## what is Schema Interference?
#Schema Inference means Spark examines the data and attempts to determine the appropriate data types automatically.


# Define an explicit schema

In [29]:
from pyspark.sql.types import(
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

In [30]:
order_schema = StructType([StructField("order_id",IntegerType(),True),
                StructField("Customer_id", StringType(),True),
                StructField("product",StringType(),True),
                StructField("category",StringType(),True),
                StructField("amount",DoubleType(),True),
                StructField("order_status",StringType(),True)])

In [31]:
df= spark.read.csv("../data/sample_orders.csv",
                   header = True,
                   schema = order_schema)

df.printSchema()
df.show()

root
 |-- order_id: integer (nullable = true)
 |-- Customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+----------+-----------+-------+------------+
|order_id|Customer_id|   product|   category| amount|order_status|
+--------+-----------+----------+-----------+-------+------------+
|     101|       C001|    Laptop|Electronics|50000.0|   Delivered|
|     102|       C002|    Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|     Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004|    Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|     Shirt|    Fashion| 2000.0|   Cancelled|
|     106|       C006|Headphones|Electronics|   NULL|   Delivered|
|     107|       C007|     Watch|    Fashion| 5000.0|     Pending|
|     108|       C008|    Mobile|Electronics|20000.0|   Delivered|
|     108|  

In [37]:
from pyspark.sql.functions import col,sum, when

df.select([sum(when (col(c).isNull(),1).otherwise (0)).alias(c)
               for c in df.columns]).show()

+--------+-----------+-------+--------+------+------------+
|order_id|Customer_id|product|category|amount|order_status|
+--------+-----------+-------+--------+------+------------+
|       0|          0|      0|       0|     1|           1|
+--------+-----------+-------+--------+------+------------+



In [38]:
#Step 6: Handle missing values

#Option A: Drop rows containing nulls

df.dropna().show()


+--------+-----------+-------+-----------+-------+------------+
|order_id|Customer_id|product|   category| amount|order_status|
+--------+-----------+-------+-----------+-------+------------+
|     101|       C001| Laptop|Electronics|50000.0|   Delivered|
|     102|       C002| Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|  Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004| Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|  Shirt|    Fashion| 2000.0|   Cancelled|
|     107|       C007|  Watch|    Fashion| 5000.0|     Pending|
|     108|       C008| Mobile|Electronics|20000.0|   Delivered|
|     108|       C008| Mobile|Electronics|20000.0|   Delivered|
+--------+-----------+-------+-----------+-------+------------+



In [40]:
#Option B: Drop rows only when amount is missing

df.dropna(subset="amount").show()
 

+--------+-----------+--------+-----------+-------+------------+
|order_id|Customer_id| product|   category| amount|order_status|
+--------+-----------+--------+-----------+-------+------------+
|     101|       C001|  Laptop|Electronics|50000.0|   Delivered|
|     102|       C002|  Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|   Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004|  Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|   Shirt|    Fashion| 2000.0|   Cancelled|
|     107|       C007|   Watch|    Fashion| 5000.0|     Pending|
|     108|       C008|  Mobile|Electronics|20000.0|   Delivered|
|     108|       C008|  Mobile|Electronics|20000.0|   Delivered|
|     109|       C009|Keyboard|Electronics| 2500.0|        NULL|
+--------+-----------+--------+-----------+-------+------------+



In [ ]:
# Option C: Replace missing amount with zero
 
df.fillna({"amount": 0.0}).show()

+--------+-----------+----------+-----------+-------+------------+
|order_id|Customer_id|   product|   category| amount|order_status|
+--------+-----------+----------+-----------+-------+------------+
|     101|       C001|    Laptop|Electronics|50000.0|   Delivered|
|     102|       C002|    Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|     Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004|    Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|     Shirt|    Fashion| 2000.0|   Cancelled|
|     106|       C006|Headphones|Electronics|    0.0|   Delivered|
|     107|       C007|     Watch|    Fashion| 5000.0|     Pending|
|     108|       C008|    Mobile|Electronics|20000.0|   Delivered|
|     108|       C008|    Mobile|Electronics|20000.0|   Delivered|
|     109|       C009|  Keyboard|Electronics| 2500.0|        NULL|
+--------+-----------+----------+-----------+-------+------------+



In [46]:
#Step 7: Remove duplicate records

#Check duplicates

print("Total Records:", df.count())

print("Unique Order IDs:", df.select("order_id").distinct().count())

Total Records: 10
Unique Order IDs: 9


In [49]:
#Remove exact duplicate records:
df = df.dropDuplicates()
print("Records after removing duplicates:", df.count())

Records after removing duplicates: 9


In [51]:
# To deduplicate based on Order Id only
df.drop_duplicates(["order_id"]).show()

+--------+-----------+----------+-----------+-------+------------+
|order_id|Customer_id|   product|   category| amount|order_status|
+--------+-----------+----------+-----------+-------+------------+
|     101|       C001|    Laptop|Electronics|50000.0|   Delivered|
|     102|       C002|    Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|     Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004|    Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|     Shirt|    Fashion| 2000.0|   Cancelled|
|     106|       C006|Headphones|Electronics|   NULL|   Delivered|
|     107|       C007|     Watch|    Fashion| 5000.0|     Pending|
|     108|       C008|    Mobile|Electronics|20000.0|   Delivered|
|     109|       C009|  Keyboard|Electronics| 2500.0|        NULL|
+--------+-----------+----------+-----------+-------+------------+



In [57]:
#Step 8: Create a cleaned DataFrame

cleaned_df = df.drop_duplicates(["order_id"]).dropna(subset=["order_id","customer_id","amount"])
cleaned_df.show()


+--------+-----------+--------+-----------+-------+------------+
|order_id|Customer_id| product|   category| amount|order_status|
+--------+-----------+--------+-----------+-------+------------+
|     101|       C001|  Laptop|Electronics|50000.0|   Delivered|
|     102|       C002|  Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|   Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004|  Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|   Shirt|    Fashion| 2000.0|   Cancelled|
|     107|       C007|   Watch|    Fashion| 5000.0|     Pending|
|     108|       C008|  Mobile|Electronics|20000.0|   Delivered|
|     109|       C009|Keyboard|Electronics| 2500.0|        NULL|
+--------+-----------+--------+-----------+-------+------------+



In [58]:
print("Original records:", 10)
print("Cleaned records:", cleaned_df.count())

Original records: 10
Cleaned records: 8


In [ ]:
cleaned_df.write.mode("overwrite").parquet("../data/processed/orders")


In [65]:
parquet_df = spark.read.parquet("../data/processed/orders")
parquet_df.show()

+--------+-----------+--------+-----------+-------+------------+
|order_id|Customer_id| product|   category| amount|order_status|
+--------+-----------+--------+-----------+-------+------------+
|     101|       C001|  Laptop|Electronics|50000.0|   Delivered|
|     102|       C002|  Mobile|Electronics|20000.0|   Delivered|
|     103|       C003|   Shoes|    Fashion| 3000.0|     Pending|
|     104|       C004|  Tablet|Electronics|15000.0|   Delivered|
|     105|       C005|   Shirt|    Fashion| 2000.0|   Cancelled|
|     107|       C007|   Watch|    Fashion| 5000.0|     Pending|
|     108|       C008|  Mobile|Electronics|20000.0|   Delivered|
|     109|       C009|Keyboard|Electronics| 2500.0|        NULL|
+--------+-----------+--------+-----------+-------+------------+

